# Critical Input DEQN: Results Synthesis

This notebook consolidates the saved `result_learning/*.zip` artifacts into unified tables, figures, and optional stochastic steady-state diagnostics. It is designed to be run after copying all result zips into `result_learning/`.

In [ ]:
from pathlib import Path
import sys, json, shutil, subprocess

ROOT = Path.cwd()
if not (ROOT / 'scripts' / 'build_results_synthesis.py').exists():
    # Colab convention if this notebook is run from /content/econml/notebooks
    ROOT = Path('/content/econml') if Path('/content/econml').exists() else ROOT.parent

# Put all result zip archives directly here. In Colab after cloning the repo, this should be:
#   /content/econml/result_learning/*.zip
# The synthesis script reads every zip in this folder and writes outputs to OUT_DIR.
RESULT_DIR = ROOT / 'result_learning'
ZIP_DIR = RESULT_DIR
OUT_DIR = RESULT_DIR / 'synthesis_outputs'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('ZIP_DIR / RESULT_DIR:', ZIP_DIR)
print('OUT_DIR:', OUT_DIR)
zip_files = sorted(ZIP_DIR.glob('*.zip'))
print('Found result zips:', len(zip_files))
for path in zip_files[:30]:
    print('  ', path.name)
if len(zip_files) > 30:
    print('  ...')
assert zip_files, f'No result zips found. Upload/copy them to: {ZIP_DIR}'


## 1. Fast Synthesis From Existing CSV/NPZ Artifacts

This step does not retrain anything. It indexes every zip, collects saved CSV tables, builds long IRF panels from saved NPZ and CSV paths, creates policy-comparison figures, optimal-policy diagnostic figures, unified steady-state tables, natural-benchmark policy-stance tables, and derived mechanism tables.


In [ ]:
cmd = [
    sys.executable, str(ROOT / 'scripts' / 'build_results_synthesis.py'),
    '--result-dir', str(RESULT_DIR),
    '--output-dir', str(OUT_DIR),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(ROOT), check=True)

## 2. Inspect Unified Tables

In [ ]:
import pandas as pd
from IPython.display import display

tables = OUT_DIR / 'tables'
for name in [
    'ir_npz_catalog.csv',
    'combined_ir_path_summary_long.csv',
    'key_event_response_summary.csv',
    'key_event_response_summary_taylor.csv',
    'main_text_policy_mechanism_summary.csv',
    'conditional_policy_tightening_mechanism_summary.csv',
    'natural_benchmark_policy_stance_summary.csv',
    'main_text_natural_benchmark_summary.csv',
    'local_monetary_wedge_candidates.csv',
    'local_monetary_wedge_summary.csv',
    'local_monetary_wedge_paths_long.csv',
    'unified_calm_steady_references.csv',
    'unified_conditional_active_repair_references.csv',
    'unified_stochastic_regime_frequencies.csv',
    'unified_stochastic_steady_state_moments.csv',
]:
    path = tables / name
    if path.exists():
        print('\n' + '=' * 100)
        print(name, path)
        df = pd.read_csv(path)
        print(df.shape)
        display(df.head(20))
    else:
        print('Missing:', path)


## 3. Display Main Policy-Comparison Figures

In [ ]:
from IPython.display import Image, display

fig_dir = OUT_DIR / 'figures'
for path in sorted(fig_dir.rglob('*.png')):
    print(path.relative_to(fig_dir))
    display(Image(filename=str(path)))

## 4. Optional: Taylor Stochastic Steady-State Distributions

This step simulates saved Taylor-rule checkpoints, so it is heavier. The local synthesis run used a light final setting: 128 paths, 500 periods, burn-in 150, thinning 5, and 16 QMC nodes. Increase these only if you need publication-grade Monte Carlo precision.


In [ ]:
RUN_TAYLOR_SSS = False

SSS_PATHS = 128
SSS_STEPS = 500
SSS_BURNIN = 150
SSS_THIN = 5
SSS_QMC_NODES = 16
DEVICE = 'cuda'  # use 'cpu' if CUDA is unavailable
DTYPE = 'float32'

if RUN_TAYLOR_SSS:
    cmd = [
        sys.executable, str(ROOT / 'scripts' / 'build_results_synthesis.py'),
        '--result-dir', str(RESULT_DIR),
        '--output-dir', str(OUT_DIR),
        '--run-taylor-sss',
        '--device', DEVICE,
        '--dtype', DTYPE,
        '--sss-paths', str(SSS_PATHS),
        '--sss-steps', str(SSS_STEPS),
        '--sss-burnin', str(SSS_BURNIN),
        '--sss-thin', str(SSS_THIN),
        '--sss-qmc-nodes', str(SSS_QMC_NODES),
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(ROOT), check=True)
else:
    print('Taylor SSS skipped. Set RUN_TAYLOR_SSS = True to regenerate stochastic steady-state distributions.')


In [ ]:
for name in ['taylor_sss_moments.csv', 'taylor_sss_regime_frequencies.csv']:
    path = tables / name
    if path.exists():
        print('\n' + '=' * 100)
        print(name)
        display(pd.read_csv(path).head(30))

## 5. Optional: Local Monetary Wedge Inside Repair-Active States

This step does not retrain networks. It reuses saved repair-active Taylor checkpoints and applies a temporary local monetary wedge to the policy rate entering the repair KKT. It is a clean diagnostic of the mechanical channel from tighter policy to financing cost and repair investment, not a full monetary-shock DEQN equilibrium.


In [ ]:
RUN_LOCAL_MONETARY_TEST = False
LOCAL_MONETARY_HORIZON = 24

if RUN_LOCAL_MONETARY_TEST:
    cmd = [
        sys.executable, str(ROOT / 'scripts' / 'build_results_synthesis.py'),
        '--result-dir', str(RESULT_DIR),
        '--output-dir', str(OUT_DIR),
        '--run-local-monetary-test',
        '--device', DEVICE,
        '--dtype', DTYPE,
        '--sss-qmc-nodes', str(SSS_QMC_NODES),
        '--local-monetary-horizon', str(LOCAL_MONETARY_HORIZON),
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(ROOT), check=True)
else:
    print('Local monetary wedge skipped. Set RUN_LOCAL_MONETARY_TEST = True to regenerate it.')

for name in ['local_monetary_wedge_candidates.csv', 'local_monetary_wedge_summary.csv']:
    path = tables / name
    if path.exists():
        print('\n' + '=' * 100)
        print(name)
        display(pd.read_csv(path).head(30))


## 6. Zip Synthesis Outputs

In [ ]:
zip_base = RESULT_DIR / 'results_synthesis_outputs'
zip_path = shutil.make_archive(str(zip_base), 'zip', OUT_DIR)
print('Saved:', zip_path)
print('Size MB:', Path(zip_path).stat().st_size / 1e6)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as exc:
    print('Download helper skipped:', exc)